# How the 3D segmentation works

In a z-stack the nuclear envelope is a closed bright surface around a dark inside, so a nucleus is a **void** (a
2-cycle) of the superlevel filtration, the 3D counterpart of the loops of the 2D segmentation. The 2D result says
where to look, so each nucleus gets its own small forest.

Steps (code in `src/nucseg/segment3d.py`):

1. **Stack.** Every slice is binned like the 2D projections, smoothed with a Gaussian of `smooth_um` along all three
   axes (in µm, so the z step matters) and scaled to the stack's 99.8th percentile.
2. **Crops with caps.** For each 2D nucleus (the *seed*): its bounding box plus `margin_um`, all slices, and a cap
   as bright as the brightest crop voxel below the first and above the last slice. A stack is often only about as
   deep as a nucleus, and the top and bottom of the envelope are faint (axial blur), so without caps many nuclei
   would not close. For a 2D nucleus cut by the image edge, a bright wall on that side of the crop plays the role
   of the 2D `border_frame`.
3. **Void.** A cubical persistence forest on the negated crop. For every bar with lifetime ≥ `min_lifetime` the
   representative at `cycle_position` of the bar is reconstructed and its inside filled. The nucleus is the void
   covering most of the seed (the 2D region placed on the centre slice of its projection window). The longest bar
   is usually a small pocket next to a cap, so it is not used.
4. **Smoothing.** A Gaussian of `mask_smooth_um` on the inside, thresholded at one half.

A nucleus whose void reaches the first or last slice is closed by a cap (`bottom_cap`, `top_cap`); its height is
then a lower bound. The 3D step only searches near the 2D nuclei, so it can lose a nucleus but not find a new one.

This notebook looks at one stack: the first TIFF in `data/`, or the synthetic example.

In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage

import nucseg
from nucseg import plots
from nucseg.segment2d import reference_image
from nucseg.surfaces import plot_3d

In [ ]:
DATA_DIR = Path("data")
CHANNEL = None
UM_PER_PIXEL = None

paths = nucseg.find_stacks([DATA_DIR]) if DATA_DIR.exists() else []
path = paths[0] if paths else nucseg.write_synthetic_stack(Path("example_data") / "synthetic.tif")
stack = nucseg.read_stack(path, channel=CHANNEL, um_per_pixel=UM_PER_PIXEL)
z_step = "unknown" if stack.z_step_um is None else f"{stack.z_step_um:.3f} µm"
print(f"{stack.name}: {stack.data.shape[0]} slices of {stack.data.shape[1]} x {stack.data.shape[2]} px, "
      f"{stack.um_per_pixel:.4f} µm/px, z step {z_step}")

In [ ]:
parameters = nucseg.Parameters(z_first=None, z_last=None, bin_factor=4, border_frame=True)
parameters_3d = nucseg.Parameters3D()
segmentation = nucseg.segment_2d(stack, parameters)
result = nucseg.segment_3d(stack, segmentation, parameters_3d)
print(f"{len(segmentation.nuclei)} 2D nuclei, {sum(n.found for n in result.nuclei)} closed voids, "
      f"z step {result.z_step_um:.2f} µm")

## Nuclei

- `lifetime_3d`: lifetime of the void, normalized like in 2D but relative to the whole stack.
- `seed_overlap`: fraction of the 2D region (on the centre slice of its window) inside the void.
- `volume_um3` (smoothed mask) and `volume_raw_um3`; `z_first` … `z_last`, `height_um`.
- `z_largest`, `largest_section_um2`: the slice with the largest section, to compare with `z_window` and
  `area_2d_um2`.
- `bottom_cap`, `top_cap`: the void reaches the first or last slice, so a cap closes it.

In [ ]:
pd.DataFrame(result.table()).set_index("label")

## All slices

Outlines of the smoothed masks on every slice; dashed outlines belong to nuclei closed by a cap. The dotted white
outline is the 2D seed on the centre slice of its projection window. Each slice is shown up to its own 99.8th
percentile.

In [ ]:
plots.plot_sections(result, title=stack.name)
plt.show()

## Orthogonal sections

xy, xz and yz sections through the centroid of a nucleus. Lime: smoothed, orange: raw. The dashed red lines mark the
ends of the stack (the caps).

In [ ]:
for nucleus in [n for n in result.nuclei if n.found][:2]:
    plots.plot_orthogonal(result, nucleus.label)
    plt.show()

## 3D surfaces

The surface of each nucleus is the voxel boundary of its smoothed mask (a cubical 2-cycle) after a few Taubin
smoothing steps.

In [ ]:
fig = plot_3d(result, z_exaggeration=2.0)
fig.update_layout(height=700)
fig.show()

## 2D outlines against 3D voids

Each 2D outline against the **3D footprint** (the 3D mask projected along z). Green: inside both; magenta: inside the
2D outline only; cyan: inside the footprint only; red: a 2D nucleus without a 3D void.

On real ER stacks the disagreement is mostly one-sided: the 2D outline runs on the bright envelope ring, while the
3D void ends at its inner edge, so most nuclei have a thin magenta rim. Larger differences come from flat nuclei
(towards the rim the top and bottom sheets of the envelope merge under the axial blur), from nuclei at the image
edge, and from bright ER channels running through a nucleus.

In [ ]:
_, h, w = result.image.shape
rows = []
both, only_2d, only_3d = (np.zeros((h, w), bool) for _ in range(3))
for n in result.nuclei:
    seed = n.seed.mask((h, w))
    footprint = n.full_mask(result.image.shape).any(axis=0) if n.found else np.zeros((h, w), bool)
    both |= seed & footprint
    only_2d |= seed & ~footprint & n.found
    only_3d |= footprint & ~seed
    if n.found:
        rows.append({"label": n.label, "border": n.border,
                     "area_2d_um2": seed.sum() * result.um_per_bin**2,
                     "footprint_um2": footprint.sum() * result.um_per_bin**2,
                     "iou": (seed & footprint).sum() / (seed | footprint).sum()})

gray = np.clip(reference_image(segmentation), 0, 1)
rgb = np.repeat(gray[..., None], 3, axis=2)
for mask, color in ((both, (0.2, 0.85, 0.2)), (only_2d, (1.0, 0.1, 0.9)), (only_3d, (0.1, 0.9, 1.0))):
    rgb[mask] = 0.35 * rgb[mask] + 0.65 * np.array(color)
s = result.um_per_bin
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(rgb, extent=(0, w * s, h * s, 0))
for n in result.nuclei:
    if not n.found:
        plots._contour(ax, n.seed.mask((h, w)), s, "red")
ax.set(xlabel="x (µm)", ylabel="y (µm)", title=stack.name)
plt.show()
pd.DataFrame(rows).set_index("label").round(2)

## Sensitivity to the z step

The z step in the file is usually the stage step. The refractive-index mismatch between immersion medium and sample
stretches or squashes real distances along z, often by 10-20 %. The z step changes volumes and heights directly; it
also changes the z smoothing of the stack (`smooth_um` is in µm), so the voids themselves can change slightly. Set
`Parameters3D(z_step_um=...)` to the corrected value.

In [ ]:
rows = []
for factor in (0.8, 1.0, 1.2):
    step = factor * stack.z_step_um
    scaled = nucseg.segment_3d(stack, segmentation, replace(parameters_3d, z_step_um=step))
    for row in scaled.table():
        rows.append({"z_step_um": round(step, 3), "label": row["label"], "volume_um3": row.get("volume_um3"),
                     "height_um": row.get("height_um")})
pd.DataFrame(rows).pivot(index="label", columns="z_step_um", values=["volume_um3", "height_um"])